# Intervals by chat: are the sessions really independent?

Every interval so far resamples conversation sessions. Sessions from the same chat share a person and recurring topics, so they are not independent, and the test set is concentrated: 67 test sessions come from only 18 chats, and the 3 largest hold 63% of my test replies. Here the main results are recomputed resampling whole chats (a cluster bootstrap): every bootstrap sample draws chats with replacement and takes all their sessions. Point estimates don't change, only the intervals. The by-session intervals are printed next to them and reproduce the published ones.

`scripts/eval_by_chat.py` reads only the saved per-example numbers, except the style test, which re-applies the frozen classifiers to the saved generations.

## Prediction (written before running)

Fede: the signal survives, although we don't know by how much. Relevance is the candidate to cross zero. The intervals will get wider, which is bad because it means less certainty, but not twice as wide; it depends on how much data each chat has.

Claude: the signal survives because it is large; relevance may cross zero because its lower bound is already close (0.09); paired differences barely change, because they compare the same replies and the chat effect cancels.

In [1]:
import subprocess
from pathlib import Path
ROOT = Path("..").resolve()
p = subprocess.run("HF_HUB_DISABLE_PROGRESS_BARS=1 uv run python scripts/eval_by_chat.py --transformer --out experiments/results/by_chat.json",
                   shell=True, cwd=ROOT, capture_output=True, text=True)
print("\n".join(l for l in p.stdout.splitlines() if "Warning" not in l))
assert p.returncode == 0, p.stderr[-2000:]

test: 67 sessions in 18 chats
signal_run1_5ep: 19.8 points | by session [14.7, 25.8] | by chat [8.9, 26.0]
signal_run1_seed1: 18.4 points | by session [13.3, 24.1] | by chat [7.5, 24.9]
signal_run1_seed2: 19.6 points | by session [14.2, 25.5] | by chat [8.1, 26.4]
relevance position adapter t1.0: +0.274 | by session [+0.087, +0.419] | by chat [+0.099, +0.465]
relevance position adapter1.5b_t1: +0.333 | by session [+0.193, +0.542] | by chat [+0.194, +0.502]
relevance position fewshot20_3b: +0.272 | by session [+0.070, +0.459] | by chat [+0.078, +0.452]
relevance position retrieval_valid: +0.374 | by session [+0.137, +0.640] | by chat [+0.123, +0.683]
relevance position adapter_rag3: +0.336 | by session [+0.173, +0.464] | by chat [+0.215, +0.417]
relevance gain retrieval_valid - adapter t1.0: +0.044 | by session [-0.037, +0.170] | by chat [-0.069, +0.202]
relevance gain adapter1.5b_t1 - adapter t1.0: +0.026 | by session [-0.050, +0.157] | by chat [-0.066, +0.159]
relevance gain adapter t

## Conclusion

**Prediction:** both right on the signal and on the paired differences; Fede right that the intervals don't double; both wrong that relevance would cross zero.

- **The signal survives, with a lower floor.** 19.8 points: by session [14.7, 25.8], by chat **[8.9, 26.0]**. The interval is about 50% wider and the lower bound drops by almost 6 points: part of the 19.8 depends on which chats fell in test. Seeds 1 and 2 behave the same ([7.5, 24.9] and [8.1, 26.4]).
- **Relevance doesn't cross zero.** The adapter's position is 0.27: [0.09, 0.42] by session, [0.10, 0.47] by chat. Beating the floor: +0.122, [+0.047, +0.192] by chat.
- **Style barely moves**: TF-IDF 0.96 [0.84, 1.06], mDeBERTa 0.81 [0.72, 0.89] by chat.
- **Paired differences barely move**, as expected: data curve +0.145 [+0.123, +0.172] and +0.080 [+0.066, +0.100], all layers -0.043 [-0.066, -0.024], 0.5B to 1.5B -0.277 [-0.309, -0.262].

**Reading:** the chat structure matters for the measure that compares two different groups of replies (mine against my friends'), not for measures where each chat is compared with itself. From here on the docs report the signal's interval by chat.